# Exercise: Structural health monitoring

In [ ]:
import os
from urllib.request import urlretrieve
import copy
import pandas as pd
import numpy as np
from math import floor, ceil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from scipy.interpolate import griddata

import plotly.graph_objects as go
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from cycler import cycler

# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)
torch.set_default_dtype(torch.float64)

## Introduction
In this application, we will apply the techniques we've discussed for classification in a more realistic problem setting. 

The case we consider is structural health monitoring of a bridge.

We have a collection of bridges modeled as 2D beams that all are believed to have a defect, such as a crack. 
We consider a simplified case in which the states of the bridges is either good enough that intervention is `unnecessary`, or that `demolition` is required.
The problem is that this defect can't always be spotted visually, therefore many sensors are installed, based on which we try to infer the level of damage.
The sensors are used to measure the displacement in y direction of the bridge for a given load.

Depending on the location and severity of the defect, the beam deformation will change.
In the figure below you can see an (exaggerated) example of how the beam deformation might change.

![beam_structure.png](https://surfdrive.surf.nl/files/index.php/s/u9dBHRZEyreHU6K/download)

By considering only the displacement in y direction of the sensors, we might be able to tell how severe the deformation is, and use it to predict which intervention is required.

The goal of this application is to create a model that correctly classifies the intervention as `unnecessary` or `demolition` based on this sensor data.

## Data
Let's take a look at the dataset first. 
It is a CSV file, and a convenient way to read and manipulate this file type is via the `Dataframe` of the `pandas` library. 
Printing a few lines of the dataset before performing the analysis is good practice. 
We load the dataset into a `Dataframe` from the `pandas` library and print a few rows from the top and bottom. 
The dataset consists of a collection of displacement fields of the bridges. 
- **sample** refers to the bridge considered.
- **node** refers to the sensor number.
- **x** is the x-coordinate of the sensor.
- **y** is the y-coordinate of the sensor.
- **dy** is the measured displacement. (Note: this includes measurement noise)
- **intervention** is the class the bridge is classified to, either `unnecessary` or `demolition`.

We have a total of 400 bridges, samples 0-199 are of class `unnecessary`, and samples 800-999 are of class `demolition`. For each sample, there are 712 locations in which the displacements have been measured, as can be seen from the number of nodes. 


## General coding & debugging tips
- For a pandas dataframe, you can use `df.iloc[::N]` to select every N'th row, and `df["name"]` to select a named column. 
- For numpy arrays and pytorch tensors you can use `x[a:b, c:d]` to select rows `a` to `b` and columns `c` to `d` from `x`.  
- If you run into problems with torch tensors being the wrong shape, print the shapes of all tensors involved using `print(x.shape)`.
- Torch batches are obtained by looping over a DataLoader, e.g.: `for x, t in test_loader:` where the batch `x` can then be passed to functions `y = model.classify(x)`. Note that `y` is then also a batch of outputs.

In [ ]:
# Download the dataset (if necessary)
url = "https://surfdrive.surf.nl/s/eo3qHBomcpTtmbQ/download"
filename = "classification-data_realistic.csv"

if not os.path.isfile(filename):
    print(f"Downloading {filename}...")
    urlretrieve(url, filename)

# Load the dataset that contains the (inherently noisy) observations
df = pd.read_csv(filename)
df.head()

In [ ]:
df.tail()

### Data visualization
Let's pick a sample and visualize the bridge, all sensor locations and the y-displacement.

In [ ]:
# Select the sample for which to plot the displacement
bar_0 = df[df["sample"] == 50]

# Create grid for plotting the heatmap
grid_x, grid_y = np.mgrid[0.02:9.98:250j, 0.02:1.98:50j]
grid_z = griddata(bar_0[["x", "y"]].to_numpy(), bar_0["dy"], (grid_x, grid_y))

# plot displacement-field and nodes
fig = go.Figure()
fig.add_trace(
    go.Heatmap(
        z=grid_z.transpose(),
        x=grid_x[:, 0],
        y=grid_y[0],
        hoverinfo="skip",
        name="heatmap",
    )
)

# plot nodes
fig.add_trace(
    go.Scatter(
        x=bar_0["x"],
        y=bar_0["y"],
        mode="markers",
        marker_color="black",
        name="",
        hovertemplate="<b>Node</b>: %{text}",
        text=bar_0["node"],
    )
)

# add buttons to display different displacement fields
fig.update_layout(
    updatemenus=[
        dict(
            buttons=list(
                [
                    dict(
                        args=[
                            "z",
                            [
                                griddata(
                                    bar_0[["x", "y"]].to_numpy(),
                                    bar_0["dy"],
                                    (grid_x, grid_y),
                                ).transpose()
                            ],
                        ],
                        label="y",
                        method="restyle",
                    )
                ]
            ),
            direction="right",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.5,
            xanchor="left",
            y=1.1,
            yanchor="bottom",
            type="buttons",
            font=dict(size=13),
        ),
    ]
)

# Add annotation for button
fig.add_annotation(
    dict(
        font=dict(size=13),
        x=0.5,
        y=1.13,
        showarrow=False,
        xref="paper",
        yref="paper",
        xanchor="right",
        yanchor="bottom",
        text="Displacement: ",
    )
)

# update xaxis range and show figure
fig.update_xaxes(range=(-0.2, 10.2), constrain="domain")
fig.show()

## Two sensors
Initially, let's start by using data from only two sensors
You can pick two sensors that measure the y displacement, this will allow us to visualize our data well. 
You can find the number of the sensor by hovering over the above plot. The following section shows which sensors you picked.

In [ ]:
# define measurement locations, get corresponding coord
# ----------------------------------------
measure_locs = [29, 2]  # <- fill in the indices of the 2 sensors you select
# ----------------------------------------

measure_coords = np.array(
    [bar_0[bar_0["node"] == loc][["x", "y"]].to_numpy() for loc in measure_locs]
).squeeze(1)

bar = df[df["sample"] == 1]
grid_x, grid_y = np.mgrid[0.02:9.98:250j, 0.02:1.98:50j]

fig = go.Figure()
# plot measurement locations
fig.add_trace(
    go.Scatter(
        x=bar_0["x"],
        y=bar_0["y"],
        mode="markers",
        marker_size=4,
        marker_color="gray",
        name="",
        hovertemplate="<b>Node</b>: %{text}",
        text=bar_0["node"],
    )
)


fig.add_trace(
    go.Scatter(
        x=measure_coords[:, 0],
        y=measure_coords[:, 1],
        mode="markers",
        marker=dict(size=15, color="DarkSlateGrey", line=dict(width=2, color="white")),
        hovertemplate="<b>Node</b>: %{text}",
        text=measure_locs,
        name="",
    )
)
fig.update_layout(showlegend=False)
fig.update_xaxes(range=(-0.2, 10.2), constrain="domain")
fig.show()

With two sensors selected, now do any of the required data pre-processing to format your data, and then try to plot how the data looks like for your two chosen sensors. 
Observe how your plots change when you change to different sensors.

Note: We've added some code and hints for a possible solution. 
However, feel free to ignore this code and implement your own instead!

Hint: 
- Based on the dataframe, create an array/tensor with the list of interventions that will be used as targets later. 
- Consider what label the outputs should have.

In [ ]:
# Pre-processing

# Get the total number of sensors (nodes)
num_sensors = df[df["sample"] == 0].shape[0]  # = 713

# Our inputs are the y displacements 'dy'
x = df[["dy"]].to_numpy().flatten()
# We group the data from all sensors
X = np.reshape(x, (-1, num_sensors))  # Shape: [Num_samples x num_features]

# Our inputs are the specific sensors
measurements = X[:, measure_locs]

print(f"\n Printing measurement data: \n")
for i in range(3):  # plotting the data for 3 samples
    print(
        f"Sample {i} - Sensor {measure_locs[0]}: {X[i,measure_locs[0]]:.6f}, Sensor {measure_locs[1]}: {X[i,measure_locs[1]]:.6f}"
    )

# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

Hint:
- Make a scatter plot of the target classes with respect to the two input features.
- Look at previous classification exercises for how this can be plotted.

In [ ]:
# Data visualization
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

## Simple classification model
If you're happy with the two sensors you've selected, it is time to implement a classification model.

Start by implementing a logistic regression model (without basis functions).
This will result in a linear decision boundary, and by only considering two features we can nicely visualize our model output.

Recall that in logistic regression the posterior probability of class $\mathcal{C}_1$ is given by
$$
p(\mathcal{C}_1\vert\mathbf{x}) = \sigma\left(\mathbf{w}^T\mathbf{x} + w_0\right),
$$
with $p(\mathcal{C}_2\vert\mathbf{x}) = 1 - p(\mathcal{C}_1\vert\mathbf{x})$, and $\sigma(\cdot)$ being the sigmoid function.

Hint:
- Implement functions / classes for:
    - Data normalization
    - Loss function 
    - Logistic regression model
    - Model optimization
    - DataLoaders (Splitting data into a train, validation and test set.)

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

Hint: 
- Normalize your data
- Prepare your inputs and outputs.
    - Should we add a bias term?
- Torch dataloaders expect inputs in the shape [num_samples, num_features] and targets in the shape [num_samples, num_outputs] 
- Create the dataloaders
- Initialize and train your model

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

Hint:
- Plot the result
    - Make predictions for a 2D grid of inputs in the original space
    - Plot this as a background with the plt.contourf(..) function
    - Plot the targets
    - Carefully consider when values need to be normalized and denormalized
    - Re-use functions from previous classification exercises
- Compute the accuracy of your model on the test set. 

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

It is unlikely that your linear logistic model is able to perform well in this classification problem. 
Think about the reasons why that is, and in the next section try to increase the model performance. 

## More complex classification model
Now you have complete freedom to get as good a prediction accuracy as possible.
Feel free to use as many sensors per sample and any type of classification model.

Hint:
- Select new `measure_locs` and plot your new selection, use the same plotting code as used above.

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

Hint:
- Implement a class for a new model of your choice.

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

Hint:
- Normalize your new data and create new DataLoaders.
- Perform model selection, by comparing the validation loss of models trained with different settings.

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

Hint:
- Compute the test accuracy of your best model, and compare this with the accuracy obtained from the first model.

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

## Wrap-up
Hopefully, your new model is better able to classify the bridges than the initial linear model!
Reflect on all the necessary pieces to make a classification model.
You can, for example, expect exam questions about which parts are different in classification models compared to a regression model. 

In week 4, we will discuss dimensionality reduction methods that would allow us to further increase our performance by considering the data from all sensors, without needing a separate input for each sensor!